In [ ]:
# Elektronik Spickzettel – QV 2026
# Bitte zuerst ausführen!
import numpy as np
import re, math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
import sys, os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

try:
    from calc import *
    from utils import p, fmt, make_input, make_button, make_dropdown, make_toggle, check_inputs
    from utils import EPS0, MU0, K_B, Q_E, shockley, vt
except ImportError as e:
    print("Fehler:", e)

THEORY_STYLE = widgets.Layout(width='100%', padding='0 20px 0 0')
BASE_BOX = {'min_width': '300px'}
LEFT_STYLE  = {**BASE_BOX, 'width':'60%', 'padding':'0 20px 0 0', 'border_right':'2px solid #dde'}
RIGHT_STYLE = {**BASE_BOX, 'width':'35%', 'padding':'0 0 0 20px'}
%matplotlib inline


<div style="text-align: center; margin-top: 100px;">

<h1 style="font-size: 48px; color: #003366;">Spickzettel QV 2026</h1>
<h2 style="font-size: 32px; color: #0055a5;">Elektroniker EFZ – Schweiz</h2>

<p style="font-size: 20px; margin-top: 50px;">
<strong>Erstellt am:</strong> 04.03.2026<br>
<strong>Geändert am:</strong> 09.03.2026<br> 
<strong>Version:</strong> v0.0.2<br>
<strong>Autor:</strong> Alvar Bolliger
</p>

<p style="font-size: 16px; margin-top: 30px; color: #555555;">
Hinweis: Dies ist ein persönlicher Spickzettel für die QV 2026. <br>
Interaktive Grafiken und Berechnungen sind im Notebook eingebettet.
</p>

</div>

<div style="border-left:4px solid #003366; padding:16px 24px; border-radius:6px; margin:16px 0; font-family: sans-serif;">
<h2 style="color:#003366; margin-top:0;">Inhaltsverzeichnis</h2>




</div>

In [ ]:
display(HTML('<a name="sec-e1-1"></a>'))
# ================================================================
# LAYOUT: Theorie (links) | Rechner (rechts)
# ================================================================

_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E1. Dioden & Spannungsregler\n"
        "---\n"
        "### E1.1 Diode – Kennlinie & Shockley-Gleichung\n"
        "Die **Shockley-Gleichung** beschreibt den Diodenstrom exakt:\n"
        "$$I_D = I_S \\cdot \\left(e^{\\frac{U_F}{n \\cdot U_T}} - 1\\right)$$\n"
        "| Grösse | Symbol | Typwert Si |\n"
        "|--------|--------|------------|\n"
        "| Flussspannung | $U_F$ | ≈ 0.7 V |\n"
        "| Sättigungsstrom | $I_S$ | 1…100 nA |\n"
        "| Idealitätsfaktor | $n$ | 1 (ideal) … 2 |\n"
        "| Thermospannung | $U_T = k_B T/q$ | 25.85 mV @ 27°C |\n"
        "\n"
        "**Sperr-Bereich:** $I \\approx -I_S$ (vernachlässigbar klein)\n\n"
        "**Durchbruch:** Zener-Diode → $U_Z$ konstant\n\n"
        "**Schottky-Diode:** $U_F \\approx 0.3$ V, sehr schnell (keine Speicherladung)"
    ))

# ---- Rechner E1.1 ----
_toggle_e11 = make_toggle('Berechnen:', ['ID aus UF', 'UF aus ID'])
_uf_in  = make_input('UF [V]  =', 'z.B. 0.65')
_id_in  = make_input('ID [A]  =', 'z.B. 10m')
_is_in  = make_input('IS [A]  =', 'leer = 1e-12')
_n_in   = make_input('n  [-]  =', 'leer = 1.0')
_tc_in  = make_input('T  [°C] =', 'leer = 27°C')
_btn_e11 = make_button()
_out_e11 = widgets.Output()

def _calc_e11(_=None):
    _out_e11.clear_output()
    with _out_e11:
        try:
            IS  = p(_is_in.value)  or 1e-12
            n   = p(_n_in.value)   or 1.0
            T   = p(_tc_in.value)  if _tc_in.value.strip() else 27.0
            if _toggle_e11.value == 'ID aus UF':
                UF = p(_uf_in.value)
                if UF is None: print("UF eingeben"); return
                r = diode_kennlinie(UF=UF, IS=IS, n=n, T_celsius=T)
                print(f"  ID  = {fmt(r['ID'], 'A')}")
                print(f"  VT  = {fmt(r['VT'], 'V')}")
            else:
                ID = p(_id_in.value)
                if ID is None: print("ID eingeben"); return
                r = diode_kennlinie(ID=ID, IS=IS, n=n, T_celsius=T)
                print(f"  UF  = {fmt(r['UF'], 'V')}")
                print(f"  VT  = {fmt(r['VT'], 'V')}")
        except (ValueError, ZeroDivisionError) as e:
            print("⚠", e)

_btn_e11.on_click(_calc_e11)
_toggle_e11.observe(_calc_e11, names='value')

_calc_e11_widget = widgets.VBox([
    _toggle_e11,
    widgets.VBox([_uf_in, _id_in]),
    widgets.VBox([_is_in, _n_in, _tc_in]),
    _btn_e11, _out_e11
])

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([_calc_e11_widget], layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e1-2"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E1.2 Zener-Diode & Vorwiderstand\n"
        "Die Zener-Diode hält $U_Z$ konstant (im Durchbruch).\n"
        "$$R_V = \\frac{U_{in} - U_Z}{I_Z + I_{Last}}$$\n"
        "$$P_{R_V} = (U_{in} - U_Z) \\cdot (I_Z + I_{Last})$$\n"
        "\n"
        "**Auslegungsregeln:**\n"
        "- $I_{Z,min}$ aus Datenblatt einhalten (sonst kein Durchbruch)\n"
        "- $I_{Z,max}$ nicht überschreiten ($P_{max}$ der Z-Diode)\n"
        "- Typisch: $I_Z \\approx 10\\% \\ldots 20\\%$ von $I_{Last}$"
    ))

_uin_in  = make_input('Uin [V]  =', 'Eingangsspannung')
_uz_in   = make_input('UZ  [V]  =', 'Zener-Spannung')
_iz_in   = make_input('IZ  [A]  =', 'z.B. 5m')
_il_in   = make_input('ILast[A] =', 'leer = 0 A')
_rv_in   = make_input('RV  [Ω]  =', 'leer = berechnen')
_btn_e12 = make_button()
_out_e12 = widgets.Output()

def _calc_e12(_=None):
    _out_e12.clear_output()
    with _out_e12:
        try:
            IL = p(_il_in.value) or 0.0
            r  = zener_vorwiderstand(
                Uin=p(_uin_in.value), UZ=p(_uz_in.value),
                IZ=p(_iz_in.value),   ILast=IL,
                RV=p(_rv_in.value))
            _u = {'Uin':'V','UZ':'V','IZ':'A','ILast':'A','RV':'Ω','PV_RV':'W'}
            for k,v in r.items():
                if k == 'berechnet' or v is None: continue
                unit = _u.get(k,'')
                print(f"  {k} = {fmt(v, unit) if isinstance(v,(int,float)) else v}")
        except (ValueError, ZeroDivisionError) as e:
            print("⚠", e)

_btn_e12.on_click(_calc_e12)
for f in [_uin_in, _uz_in, _iz_in, _il_in, _rv_in]:
    f.observe(_calc_e12, names='value')

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_uin_in,_uz_in,_iz_in,_il_in,_rv_in,_btn_e12,_out_e12])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e1-3"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E1.3 LM317 – Verstellbarer Linearregler\n"
        "$$U_{out} = 1.25 \\cdot \\left(1 + \\frac{R_2}{R_1}\\right) + I_{adj} \\cdot R_2$$\n"
        "| Grösse | Wert |\n"
        "|--------|------|\n"
        "| $U_{ref}$ | 1.25 V (intern) |\n"
        "| $I_{adj}$ | ≈ 50 µA (vernachlässigbar) |\n"
        "| $R_1$ | typisch 240 Ω |\n"
        "| Dropout | ≈ 2…3 V |\n"
        "\n"
        "**Effizienz:** $\\eta = U_{out}/U_{in}$\n\n"
        "**Verlustleistung:** $P_V = (U_{in} - U_{out}) \\cdot I_{Last}$\n\n"
        "> LDO-Regler: Dropout < 0.5 V (z.B. LT1086)"
    ))

_r1_in   = make_input('R1 [Ω]  =', 'leer = berechnen (typ. 240)')
_r2_in   = make_input('R2 [Ω]  =', 'leer = berechnen')
_uo_in   = make_input('Uout[V] =', 'leer = berechnen')
_btn_e13 = make_button()
_out_e13 = widgets.Output()

def _calc_e13(_=None):
    _out_e13.clear_output()
    with _out_e13:
        try:
            r = lm317(R1=p(_r1_in.value), R2=p(_r2_in.value), Uout=p(_uo_in.value))
            print(f"  R1   = {fmt(r['R1'], 'Ω')}")
            print(f"  R2   = {fmt(r['R2'], 'Ω')}")
            print(f"  Uout = {fmt(r['Uout'], 'V')}")
            if r.get('warnung_Uin_min'):
                print(f"  → Uin,min ≈ {fmt(r['warnung_Uin_min'], 'V')}")
        except (ValueError, ZeroDivisionError) as e:
            print("⚠", e)

_btn_e13.on_click(_calc_e13)
for f in [_r1_in, _r2_in, _uo_in]:
    f.observe(_calc_e13, names='value')

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_r1_in,_r2_in,_uo_in,_btn_e13,_out_e13])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e2-1"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E2. Transistoren (BJT & MOSFET)\n"
        "---\n"
        "### E2.1 BJT – Bipolartransistor\n"
        "$$I_C = \\beta \\cdot I_B \\qquad I_E = I_C + I_B$$\n"
        "$$U_{BE} \\approx 0.7\\,\\text{V} \\qquad U_{CE,sat} \\approx 0.2\\,\\text{V}$$\n"
        "| Bereich | BE | BC | Anwendung |\n"
        "|---------|----|----|----------|\n"
        "| Aktiv | Flussp. | Sperrp. | Verstärker |\n"
        "| Sättigung | Flussp. | Flussp. | Schalter EIN |\n"
        "| Sperrung | Sperrp. | Sperrp. | Schalter AUS |\n"
        "\n"
        "**Basisvorwiderstand (Schalterbetrieb):**\n"
        "$$R_B = \\frac{U_{in} - U_{BE}}{I_{C,soll}/\\beta \\cdot k} \\quad k = 2 \\ldots 10$$"
    ))

_mode_e21 = make_toggle('Modus:', ['Grundgrössen', 'Basiswiderstand'])
# Grundgrössen
_ic_in = make_input('IC [A]   =', 'leer = berechnen')
_ib_in = make_input('IB [A]   =', 'leer = berechnen')
_ie_in = make_input('IE [A]   =', 'leer = berechnen')
_bt_in = make_input('β  [-]   =', 'leer = berechnen')
_uce_in = make_input('UCE [V]  =', 'optional')
# Basiswiderstand
_ui_in  = make_input('Uin [V]  =')
_ic2_in = make_input('IC [A]   =')
_bt2_in = make_input('β  [-]   =', 'leer = 100')
_k_in   = make_input('k  [-]   =', 'leer = 5')
_btn_e21 = make_button()
_out_e21 = widgets.Output()

_box_grund = widgets.VBox([_ic_in,_ib_in,_ie_in,_bt_in,_uce_in])
_box_rb    = widgets.VBox([_ui_in,_ic2_in,_bt2_in,_k_in])
_cont_e21  = widgets.VBox([_box_grund, _btn_e21, _out_e21])

def _update_e21(_=None):
    if _mode_e21.value == 'Grundgrössen':
        _cont_e21.children = [_box_grund, _btn_e21, _out_e21]
    else:
        _cont_e21.children = [_box_rb, _btn_e21, _out_e21]

def _calc_e21(_=None):
    _out_e21.clear_output()
    with _out_e21:
        try:
            if _mode_e21.value == 'Grundgrössen':
                r = bjt_basis(IC=p(_ic_in.value), IB=p(_ib_in.value),
                               IE=p(_ie_in.value), beta=p(_bt_in.value),
                               UCE=p(_uce_in.value))
                _u = {'IC':'A','IB':'A','IE':'A','UCE':'V','UBE':'V','PV':'W'}
                for k,v in r.items():
                    if k in ('arbeitspunkt',) and v: print(f"  → {v}"); continue
                    if v is None or isinstance(v,str): continue
                    print(f"  {k} = {fmt(v, _u.get(k,''))}")
                if r.get('arbeitspunkt'): print(f"  Bereich: {r['arbeitspunkt']}")
            else:
                beta = p(_bt2_in.value) or 100
                k    = p(_k_in.value)  or 5
                r = bjt_basiswiderstand(
                    Uin=p(_ui_in.value), IC_soll=p(_ic2_in.value),
                    beta=beta, k_ueberst=k)
                print(f"  RB = {fmt(r['RB'], 'Ω')}")
                print(f"  IB = {fmt(r['RB'] and (p(_ui_in.value)-0.7)/r['RB'], 'A')}")
                if r.get('P_RB'): print(f"  P_RB = {fmt(r['P_RB'], 'W')}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e21.observe(_update_e21, names='value')
_btn_e21.on_click(_calc_e21)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e21, _cont_e21])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e2-2"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E2.2 MOSFET (N-Kanal Enhancement)\n"
        "Steuergrösse: $U_{GS}$ – **kein** Gatestrom ($I_G \\approx 0$)\n"
        "| Bereich | Bedingung | Drainstrom |\n"
        "|---------|-----------|------------|\n"
        "| Sperrung | $U_{GS} < U_{th}$ | $I_D \\approx 0$ |\n"
        "| Ohmscher B. | $U_{DS} < U_{GS}-U_{th}$ | $\\propto U_{DS}$ |\n"
        "| Sättigung | $U_{DS} \\geq U_{GS}-U_{th}$ | $= \\frac{K}{2}(U_{GS}-U_{th})^2$ |\n"
        "\n"
        "**Verlust im eingeschalteten Zustand:**\n"
        "$$P_V = R_{DS(on)} \\cdot I_D^2 \\quad \\text{(Ohmscher Bereich)}$$"
    ))

_ugs_in  = make_input('UGS [V]  =')
_uth_in  = make_input('Uth [V]  =', 'leer = 2.5 V')
_k_in2   = make_input('K [A/V²] =', 'leer = 0.1')
_uds_in  = make_input('UDS [V]  =', 'optional')
_reg_tog = make_toggle('Bereich:', ['auto', 'sättigung', 'ohmscher'])
_btn_e22 = make_button()
_out_e22 = widgets.Output()

def _calc_e22(_=None):
    _out_e22.clear_output()
    with _out_e22:
        try:
            Uth = p(_uth_in.value) or 2.5
            K   = p(_k_in2.value) or 0.1
            reg = _reg_tog.value if _reg_tog.value != 'auto' else 'sättigung'
            r   = mosfet_ids(UGS=p(_ugs_in.value), Uth=Uth, K=K,
                              region=reg, UDS=p(_uds_in.value))
            _u = {'UGS':'V','Uth':'V','UDS':'V','IDS':'A','Uov':'V','RDS_on':'Ω','PV':'W'}
            for k,v in r.items():
                if k in ('region',): print(f"  Bereich: {v}"); continue
                if v is None: continue
                if isinstance(v, str): continue
                print(f"  {k} = {fmt(v, _u.get(k,''))}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_btn_e22.on_click(_calc_e22)
for f in [_ugs_in, _uth_in, _k_in2, _uds_in]:
    f.observe(_calc_e22, names='value')

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_ugs_in,_uth_in,_k_in2,_uds_in,_reg_tog,_btn_e22,_out_e22])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e2-3"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E2.3 Leistungshalbleiter & Thermik\n"
        "> ⚠️ Wärmewiderstands-Grundlagen → ET-Spick Kap. 10\n\n"
        "| Typ | Verlustleistung |\n"
        "|-----|----------------|\n"
        "| BJT | $P_V = U_{CE} \\cdot I_C$ |\n"
        "| MOSFET | $P_V = R_{DS(on)} \\cdot I_D^2$ |\n"
        "| IGBT | $P_V = U_{CE,sat}\\cdot I_C + E_{sw}\\cdot f_{sw}$ |\n"
        "\n"
        "**Kühlkörper-Auslegung:**\n"
        "$$R_{th,sa} \\leq \\frac{T_{j,max} - T_{amb}}{P_V} - R_{th,jc} - R_{th,cs}$$\n"
        "Typisch: $R_{th,cs} \\approx 0.2$ K/W (mit Wärmeleitpaste)"
    ))

_mode_e23 = make_toggle('Typ:', ['BJT', 'MOSFET'])
_uce_e23  = make_input('UCE [V]  =', 'BJT: Kollektor-Emitter')
_ic_e23   = make_input('IC  [A]  =')
_rds_e23  = make_input('RDS_on[Ω]=', 'MOSFET: Einschaltwiderstand')
_tj_e23   = make_input('Tj,max[°C]=', 'leer = 150°C')
_ta_e23   = make_input('Tamb [°C]=', 'leer = 25°C')
_rjc_e23  = make_input('Rjc[K/W] =', 'aus Datenblatt')
_btn_e23  = make_button()
_out_e23  = widgets.Output()

def _calc_e23(_=None):
    _out_e23.clear_output()
    with _out_e23:
        try:
            Tj  = p(_tj_e23.value)  or 150
            Ta  = p(_ta_e23.value)  or 25
            if _mode_e23.value == 'BJT':
                r = leistungshalbleiter(UCE=p(_uce_e23.value), IC=p(_ic_e23.value),
                                         Tj_max=Tj, Tamb=Ta, Rjc=p(_rjc_e23.value))
            else:
                r = leistungshalbleiter(RDS_on=p(_rds_e23.value), IC=p(_ic_e23.value),
                                         Tj_max=Tj, Tamb=Ta, Rjc=p(_rjc_e23.value))
            _u = {'PV':'W','UCE':'V','IC':'A','RDS_on':'Ω',
                  'Rth_ges_max':'K/W','Rsa_max':'K/W'}
            for k,v in r.items():
                if k in ('modus','kuehlkoerper','warnung'):
                    print(f"  {k}: {v}"); continue
                if v is None or isinstance(v,bool): continue
                if isinstance(v,str): continue
                print(f"  {k} = {fmt(v, _u.get(k,''))}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e23.observe(_calc_e23, names='value')
_btn_e23.on_click(_calc_e23)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e23,_uce_e23,_ic_e23,_rds_e23,_tj_e23,_ta_e23,_rjc_e23,_btn_e23,_out_e23])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e3-1"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E3. Operationsverstärker (OPV)\n"
        "---\n"
        "### E3.1 Grundschaltungen\n"
        "**Goldene Regeln (ideal):** $U_+ = U_-$, $I_{in} = 0$\n\n"
        "| Schaltung | Verstärkung | Eingangsimpedanz |\n"
        "|-----------|-------------|------------------|\n"
        "| Invertierend | $v = -R_f/R_{in}$ | $R_{in}$ |\n"
        "| Nicht-inv. | $v = 1 + R_f/R_2$ | $\\to \\infty$ |\n"
        "| Impedanzwandler | $v = 1$ | $\\to \\infty$ |\n"
        "| Differenz | $v = R_f/R_1$ | $R_1$ |\n"
        "\n"
        "**Kompensationswiderstand** (inv.):\n"
        "$$R_{komp} = R_{in} \\| R_f$$"
    ))

_mode_e31 = make_toggle('Schaltung:', ['Invertierend', 'Nicht-Inv.'])
_rf_in  = make_input('Rf  [Ω] =', 'Gegenkopplungswid.')
_rin_in = make_input('Rin [Ω] =', 'Eingangswiderstand (inv.)')
_r2_in  = make_input('R2  [Ω] =', 'Unterer Wid. (n.i.)')
_ui_e31 = make_input('Uin [V] =', 'optional')
_uo_e31 = make_input('Uout[V] =', 'leer = berechnen')
_btn_e31 = make_button()
_out_e31 = widgets.Output()

def _calc_e31(_=None):
    _out_e31.clear_output()
    with _out_e31:
        try:
            if _mode_e31.value == 'Invertierend':
                r = opv_invertierend(Uin=p(_ui_e31.value), Rf=p(_rf_in.value),
                                      Rin=p(_rin_in.value), Uout=p(_uo_e31.value))
            else:
                r = opv_nicht_invertierend(Uin=p(_ui_e31.value), Rf=p(_rf_in.value),
                                            R2=p(_r2_in.value), Uout=p(_uo_e31.value))
            _u = {'Rf':'Ω','Rin':'Ω','R2':'Ω','Uin':'V','Uout':'V','Rkomp':'Ω'}
            for k,v in r.items():
                if v is None: continue
                if k == 'v_dB' and v is not None: print(f"  v_dB = {v:.2f} dB"); continue
                if k == 'impedanzwandler':
                    if v: print("  → Impedanzwandler"); continue
                if isinstance(v, bool): continue
                print(f"  {k} = {fmt(v, _u.get(k,'')) if isinstance(v,(int,float)) else v}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e31.observe(_calc_e31, names='value')
_btn_e31.on_click(_calc_e31)
for f in [_rf_in,_rin_in,_r2_in,_ui_e31,_uo_e31]:
    f.observe(_calc_e31, names='value')

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e31,_rf_in,_rin_in,_r2_in,_ui_e31,_uo_e31,_btn_e31,_out_e31])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e3-2"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E3.2 GBP & Slew Rate\n"
        "**Gain-Bandwidth-Product (GBP):**\n"
        "$$\\text{GBP} = |v| \\cdot f_{-3\\,\\text{dB}} = \\text{const.}$$\n"
        "Höhere Verstärkung → niedrigere Bandbreite\n\n"
        "**Slew Rate (SR):**\n"
        "$$SR = \\frac{\\Delta U_{out}}{\\Delta t} \\left[\\frac{\\text{V}}{\\mu\\text{s}}\\right]$$\n"
        "Maximale Aussteuerungsfrequenz:\n"
        "$$f_{max} = \\frac{SR}{2\\pi \\cdot \\hat{U}_{out}}$$\n"
        "| OPV | GBP | SR |\n"
        "|-----|-----|----|\n"
        "| LM741 | 1 MHz | 0.5 V/µs |\n"
        "| TL071 | 3 MHz | 13 V/µs |\n"
        "| LM318 | 15 MHz | 70 V/µs |"
    ))

_mode_e32 = make_toggle('Berechnung:', ['GBP', 'Slew Rate'])
_gbp_in  = make_input('GBP [Hz] =', 'z.B. 1M')
_v_in    = make_input('v   [-]  =', 'Verstärkungsbetrag')
_sr_in   = make_input('SR [V/µs]=', 'aus Datenblatt')
_uo_e32  = make_input('Uout[V]  =', 'Ausgangsamplitude')
_f_e32   = make_input('f   [Hz] =', 'optional: Frequenz prüfen')
_btn_e32 = make_button()
_out_e32 = widgets.Output()

def _calc_e32(_=None):
    _out_e32.clear_output()
    with _out_e32:
        try:
            if _mode_e32.value == 'GBP':
                gbp = p(_gbp_in.value)
                v   = p(_v_in.value)
                if gbp and v:
                    r = opv_gbp(gbp, v)
                    print(f"  f_-3dB = {fmt(r['f_3dB'], 'Hz')}")
                    print(f"  GBP    = {fmt(r['GBP'], 'Hz')}")
            else:
                sr   = p(_sr_in.value)
                uout = p(_uo_e32.value)
                if sr and uout:
                    r = opv_slew_rate(sr, uout, f=p(_f_e32.value))
                    print(f"  f_max  = {fmt(r['f_max'], 'Hz')}")
                    if 'f' in r:
                        print(f"  Bei f={fmt(r['f'],'Hz')}: Uout_max = {fmt(r['Uout_max_bei_f'],'V')}")
                        print(f"  OK: {r['ok']}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_btn_e32.on_click(_calc_e32)
_mode_e32.observe(_calc_e32, names='value')

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e32,_gbp_in,_v_in,_sr_in,_uo_e32,_f_e32,_btn_e32,_out_e32])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e3-3"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "### E3.3 Schmitt-Trigger & Integrator\n"
        "**Invertierender Schmitt-Trigger** (Mitkopplung):\n"
        "$$U_{OTP} = U_{sat+} \\cdot \\frac{R_1}{R_1+R_f}$$\n"
        "$$U_{UTP} = U_{sat-} \\cdot \\frac{R_1}{R_1+R_f}$$\n"
        "$$\\Delta U_{Hyst} = U_{OTP} - U_{UTP}$$\n"
        "\n"
        "**OPV-Integrator:**\n"
        "$$U_{out}(t) = -\\frac{1}{RC}\\int U_{in}\\,dt$$\n"
        "Für DC-Eingang: $U_{out} = -\\frac{U_{in}}{RC}\\cdot t$ (Rampe)\n\n"
        "**OPV-Differentiator:**\n"
        "$$U_{out} = -RC\\cdot\\frac{dU_{in}}{dt}$$\n"
        "Nadelimpuls bei Sprung – rauschempfindlich!"
    ))

_mode_e33 = make_toggle('Modus:', ['Schmitt-Trigger', 'Integrator'])
# Schmitt-Trigger
_rf_e33  = make_input('Rf [Ω]   =', 'leer = berechnen')
_r1_e33  = make_input('R1 [Ω]   =', 'leer = berechnen')
_sp_e33  = make_input('Usat+ [V]=', 'leer = 13.5 V')
_sn_e33  = make_input('Usat- [V]=', 'leer = -13.5 V')
_uotp_e33= make_input('Uotp [V] =', 'optional: Schaltschwelle')
# Integrator
_ri_e33  = make_input('R  [Ω]   =')
_ci_e33  = make_input('C  [F]   =')
_ui_e33  = make_input('Uin [V]  =', 'DC-Eingang')
_ti_e33  = make_input('t  [s]   =')
_btn_e33 = make_button()
_out_e33 = widgets.Output()

_box_st = widgets.VBox([_rf_e33,_r1_e33,_sp_e33,_sn_e33,_uotp_e33])
_box_int= widgets.VBox([_ri_e33,_ci_e33,_ui_e33,_ti_e33])
_cont_e33 = widgets.VBox([_box_st, _btn_e33, _out_e33])

def _update_e33(_=None):
    if _mode_e33.value == 'Schmitt-Trigger':
        _cont_e33.children = [_box_st, _btn_e33, _out_e33]
    else:
        _cont_e33.children = [_box_int, _btn_e33, _out_e33]

def _calc_e33(_=None):
    _out_e33.clear_output()
    with _out_e33:
        try:
            if _mode_e33.value == 'Schmitt-Trigger':
                Usp = p(_sp_e33.value) or  13.5
                Usn = p(_sn_e33.value) or -13.5
                r = schmitt_trigger_inv(Rf=p(_rf_e33.value), R1=p(_r1_e33.value),
                                         Usat_pos=Usp, Usat_neg=Usn,
                                         Uotp=p(_uotp_e33.value))
                print(f"  Uotp      = {fmt(r['Uotp'], 'V')}")
                print(f"  Uutp      = {fmt(r['Uutp'], 'V')}")
                print(f"  Hysterese = {fmt(r['hysterese'], 'V')}")
                if r.get('Rf'): print(f"  Rf = {fmt(r['Rf'], 'Ω')}")
                if r.get('R1'): print(f"  R1 = {fmt(r['R1'], 'Ω')}")
            else:
                r = opv_integrator(R=p(_ri_e33.value), C=p(_ci_e33.value),
                                    Uin=p(_ui_e33.value), t=p(_ti_e33.value))
                print(f"  τ     = {fmt(r['tau'], 's')}")
                print(f"  f_int = {fmt(r['f_int'], 'Hz')}")
                if 'Uout' in r:
                    print(f"  Uout  = {fmt(r['Uout'], 'V')}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e33.observe(_update_e33, names='value')
_btn_e33.on_click(_calc_e33)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e33, _cont_e33])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e4-1"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E4. Oszillatoren & Timer\n"
        "---\n"
        "### E4.1 Schwingbedingung (Barkhausen)\n"
        "$$|k \\cdot v| = 1 \\quad \\text{und} \\quad \\varphi_{ges} = 0°$$\n"
        "### E4.2 Wien-Brücken-Oszillator\n"
        "$$f_0 = \\frac{1}{2\\pi RC} \\quad | \\quad v_{min} = 3 \\quad | \\quad R_f = 2R_1$$\n"
        "### E4.2 RC-Phasenschieber-Oszillator\n"
        "$$f_0 = \\frac{1}{2\\pi RC\\sqrt{6}} \\quad | \\quad v_{min} = 29$$\n"
        "3 RC-Glieder à 60° Phasenverschiebung\n"
        "### E4.3 555 Timer – Astabil\n"
        "$$f = \\frac{1.44}{(R_A + 2R_B)\\cdot C}$$\n"
        "$$D = \\frac{R_A+R_B}{R_A+2R_B} \\quad t_{high}=0.693(R_A+R_B)C$$\n"
        "### E4.3 555 Timer – Monostabil\n"
        "$$t_p = 1.1 \\cdot R \\cdot C$$"
    ))

_mode_e4 = make_toggle('Schaltung:', ['Wien-Osc', 'RC-Phase', '555 Astab.', '555 Monos.'])
_r_e4   = make_input('R [Ω]   =', 'leer = berechnen')
_c_e4   = make_input('C [F]   =', 'leer = berechnen')
_f_e4   = make_input('f0 [Hz] =', 'leer = berechnen')
_ra_e4  = make_input('RA [Ω]  =')
_rb_e4  = make_input('RB [Ω]  =')
_c2_e4  = make_input('C  [F]  =')
_tp_e4  = make_input('tp [s]  =', 'leer = berechnen')
_btn_e4 = make_button()
_out_e4 = widgets.Output()

_box_wien = widgets.VBox([_r_e4, _c_e4, _f_e4])
_box_rc   = widgets.VBox([_r_e4, _c_e4, _f_e4])
_box_555a = widgets.VBox([_ra_e4, _rb_e4, _c2_e4])
_box_555m = widgets.VBox([_r_e4, _c_e4, _tp_e4])
_cont_e4  = widgets.VBox([_box_wien, _btn_e4, _out_e4])

def _update_e4(_=None):
    m = _mode_e4.value
    if m == 'Wien-Osc':   _cont_e4.children = [_box_wien, _btn_e4, _out_e4]
    elif m == 'RC-Phase':  _cont_e4.children = [_box_wien, _btn_e4, _out_e4]
    elif m == '555 Astab.':_cont_e4.children = [_box_555a, _btn_e4, _out_e4]
    else:                  _cont_e4.children = [_box_555m, _btn_e4, _out_e4]

def _calc_e4(_=None):
    _out_e4.clear_output()
    with _out_e4:
        try:
            m = _mode_e4.value
            if m == 'Wien-Osc':
                r = wien_oszillator(R=p(_r_e4.value), C=p(_c_e4.value), f0=p(_f_e4.value))
                print(f"  f0   = {fmt(r['f0'], 'Hz')}")
                print(f"  R    = {fmt(r['R'], 'Ω')}")
                print(f"  C    = {fmt(r['C'], 'F')}")
                print(f"  v    ≥ {r['v_min']}  (Rf = 2·R1)")
            elif m == 'RC-Phase':
                r = rc_phasenschieber_osz(R=p(_r_e4.value), C=p(_c_e4.value), f0=p(_f_e4.value))
                print(f"  f0   = {fmt(r['f0'], 'Hz')}")
                print(f"  R    = {fmt(r['R'], 'Ω')}")
                print(f"  C    = {fmt(r['C'], 'F')}")
                print(f"  v    ≥ {r['v_min']}")
            elif m == '555 Astab.':
                r = timer_555_astabil(RA=p(_ra_e4.value), RB=p(_rb_e4.value), C=p(_c2_e4.value))
                print(f"  f      = {fmt(r['f'], 'Hz')}")
                print(f"  T      = {fmt(r['T'], 's')}")
                print(f"  D      = {r['D']*100:.1f} %")
                print(f"  t_high = {fmt(r['t_high'], 's')}")
                print(f"  t_low  = {fmt(r['t_low'], 's')}")
            else:
                r = timer_555_monostabil(R=p(_r_e4.value), C=p(_c_e4.value), tp=p(_tp_e4.value))
                print(f"  tp = {fmt(r['tp'], 's')}")
                print(f"  R  = {fmt(r['R'], 'Ω')}")
                print(f"  C  = {fmt(r['C'], 'F')}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e4.observe(_update_e4, names='value')
_btn_e4.on_click(_calc_e4)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e4, _cont_e4])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
display(HTML('<a name="sec-e5-1"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E5. Regelungstechnik\n"
        "---\n"
        "### E5.1 PID-Regler\n"
        "$$y(t) = K_P\\left[e(t) + \\frac{1}{T_I}\\int e\\,dt + T_D\\frac{de}{dt}\\right]$$\n"
        "| Typ | Bleibende Abw. | Wirkung |\n"
        "|-----|---------------|--------|\n"
        "| P  | Ja | Schnell, einfach |\n"
        "| I  | Nein | Träge, eliminiert Abw. |\n"
        "| D  | Ja | Schnell, rauschempf. |\n"
        "| PI | Nein | Gut für träge Systeme |\n"
        "| PID| Nein | Optimal |\n"
        "\n"
        "### E5.2 PT1-Sprungantwort\n"
        "$$x(t) = K_S \\cdot w \\cdot (1 - e^{-t/T})$$\n"
        "- Bei $t = T$: $x \\approx 63.2\\%$ des Endwerts\n"
        "- Bei $t = 5T$: $x \\approx 99.3\\%$ (eingeschwungen)\n"
        "\n"
        "**Stabilitätskriterien:**\n"
        "- Phasenreserve $\\varphi_R > 45°$\n"
        "- Amplitudenreserve $A_R > 6\\,\\text{dB}$"
    ))

_mode_e5 = make_toggle('Modus:', ['PT1-Sprungantwort', 'PID-Ausgang'])
# PT1
_ks_in   = make_input('KS [-]   =', 'Verstärkung Strecke')
_T_in    = make_input('T  [s]   =', 'Zeitkonstante')
_t_in    = make_input('t  [s]   =', 'Zeitpunkt')
_w_in    = make_input('w  [-]   =', 'leer = 1.0 (Sprung)')
# PID
_kp_in   = make_input('KP [-]   =', 'leer = 1.0')
_ti_in   = make_input('TI [s]   =', 'leer = kein I-Anteil')
_td_in   = make_input('TD [s]   =', 'leer = kein D-Anteil')
_e_in    = make_input('e  [-]   =', 'aktuelle Regelabw.')
_ep_in   = make_input('e_prev[-]=', 'leer = 0')
_es_in   = make_input('e_sum[-] =', 'leer = 0')
_dt_in   = make_input('dt [s]   =', 'Zeitschritt')
_btn_e5  = make_button()
_out_e5  = widgets.Output()

_box_pt1 = widgets.VBox([_ks_in,_T_in,_t_in,_w_in])
_box_pid = widgets.VBox([_kp_in,_ti_in,_td_in,_e_in,_ep_in,_es_in,_dt_in])
_cont_e5 = widgets.VBox([_box_pt1, _btn_e5, _out_e5])

def _update_e5(_=None):
    if _mode_e5.value == 'PT1-Sprungantwort':
        _cont_e5.children = [_box_pt1, _btn_e5, _out_e5]
    else:
        _cont_e5.children = [_box_pid, _btn_e5, _out_e5]

def _calc_e5(_=None):
    _out_e5.clear_output()
    with _out_e5:
        try:
            if _mode_e5.value == 'PT1-Sprungantwort':
                w = p(_w_in.value) or 1.0
                r = pt1_sprungantwort(KS=p(_ks_in.value), T=p(_T_in.value),
                                       t=p(_t_in.value), w=w)
                print(f"  x(t)     = {fmt(r['x_t'])}")
                print(f"  x_end    = {fmt(r['x_end'])}")
                print(f"  % Endw.  = {r['pct_von_endwert']:.1f} %")
                print(f"  t_63%    = {fmt(r['t_63pct'], 's')}")
                print(f"  t_99%    = {fmt(r['t_99pct'], 's')}")
            else:
                KP = p(_kp_in.value) or 1.0
                e_prev = p(_ep_in.value) or 0.0
                e_sum  = p(_es_in.value) or 0.0
                r = pid_ausgabe(e=p(_e_in.value) or 0, e_prev=e_prev,
                                e_sum=e_sum, dt=p(_dt_in.value) or 0.1,
                                KP=KP, TI=p(_ti_in.value), TD=p(_td_in.value))
                print(f"  y (Stell.) = {fmt(r['y'])}")
                print(f"  P-Anteil   = {fmt(r['P'])}")
                print(f"  I-Anteil   = {fmt(r['I'])}")
                print(f"  D-Anteil   = {fmt(r['D'])}")
                print(f"  e_sum_neu  = {fmt(r['e_sum'])}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e5.observe(_update_e5, names='value')
_btn_e5.on_click(_calc_e5)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e5, _cont_e5])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))


In [ ]:
# ── Interaktives PT1-Diagramm ──────────────────────────────────
import numpy as np, matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

_ks_plt = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='KS', readout_format='.1f')
_T_plt  = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='T [s]', readout_format='.1f')
_D_plt  = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Dämpf. D', readout_format='.1f',
                               layout=widgets.Layout(width='400px'))

_out_plt = widgets.Output()

def _plot_pt1(KS=1, T=1, D=1):
    _out_plt.clear_output(wait=True)
    with _out_plt:
        t = np.linspace(0, 5*T, 500)
        if D == 1:  # PT1
            x = KS * (1 - np.exp(-t/T))
            label = f"PT1: KS={KS}, T={T}s"
        elif D > 1:  # Überdämpft PT2
            w0 = 1/(T)
            import cmath
            r1 = -D*w0 + w0*np.sqrt(D**2-1+0j)
            r2 = -D*w0 - w0*np.sqrt(D**2-1+0j)
            x = KS * (1 + np.real(r2/(r1-r2)*np.exp(r1*t) - r1/(r1-r2)*np.exp(r2*t)))
            label = f"PT2 überdämpft: D={D:.1f}"
        else:  # Schwingend PT2
            wd = (1/T) * np.sqrt(1 - D**2)
            x = KS * (1 - np.exp(-D/T*t)*(np.cos(wd*t) + D/np.sqrt(1-D**2)*np.sin(wd*t)))
            label = f"PT2 schwingend: D={D:.1f}"
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(t, x, 'b-', lw=2, label=label)
        ax.axhline(KS, color='r', ls='--', lw=1, label=f'Endwert={KS}')
        ax.axhline(0.632*KS, color='g', ls=':', lw=1, label='63.2%')
        ax.axvline(T, color='g', ls=':', lw=1)
        ax.set_xlabel('Zeit [s]')
        ax.set_ylabel('x(t)')
        ax.set_title('Sprungantwort')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

_w = widgets.interactive_output(_plot_pt1, {'KS':_ks_plt,'T':_T_plt,'D':_D_plt})
display(widgets.VBox([
    widgets.HTML("<b>PT1/PT2 Sprungantwort – Interaktiv</b>"),
    widgets.HBox([_ks_plt, _T_plt, _D_plt]),
    _w
]))


In [ ]:
display(HTML('<a name="sec-e6-1"></a>'))
_theory_out = widgets.Output(layout=THEORY_STYLE)
with _theory_out:
    display(Markdown(
        "---\n"
        "## E6. FFT & Signalverarbeitung\n"
        "---\n"
        "### E6.1 FFT – Kenngrößen\n"
        "$$X[k] = \\sum_{n=0}^{N-1} x[n] \\cdot e^{-j2\\pi kn/N}$$\n"
        "| Grösse | Formel |\n"
        "|--------|--------|\n"
        "| Frequenzauflösung | $\\Delta f = f_s / N$ |\n"
        "| Nyquist-Freq. | $f_{Ny} = f_s / 2$ |\n"
        "| Abtasttheorem | $f_s \\geq 2 f_{sig,max}$ |\n"
        "| Messdauer | $T = N / f_s$ |\n"
        "\n"
        "**Leakage:** Signal-Periode muss ganzzahlig ins Fenster passen, sonst Energie-Leck in Nachbarbins.\n\n"
        "**Fensterfunktionen:** Hann (allgemein), Blackman (hohe Dynamik), Flattop (Amplitudenmessung)\n\n"
        "### E6.3 ADC – Kennwerte\n"
        "$$\\text{LSB} = \\frac{U_{ref}}{2^N} \\qquad \\text{SNR} = 6.02N + 1.76\\,\\text{dB}$$"
    ))

_mode_e6 = make_toggle('Berechnung:', ['FFT-Parameter', 'ADC-Kennwerte'])
# FFT
_fs_in  = make_input('fs [Hz]  =', 'Abtastfrequenz')
_N_in   = make_input('N  [-]   =', 'Anzahl Samples (2^n)')
# ADC
_nb_in  = make_input('N_bit    =', 'z.B. 12')
_ur_in  = make_input('Uref [V] =', 'leer = 3.3 V')
_ui_e6  = make_input('Uin [V]  =', 'optional: Digitalisieren')
_btn_e6 = make_button()
_out_e6 = widgets.Output()

_box_fft = widgets.VBox([_fs_in, _N_in])
_box_adc = widgets.VBox([_nb_in, _ur_in, _ui_e6])
_cont_e6 = widgets.VBox([_box_fft, _btn_e6, _out_e6])

def _update_e6(_=None):
    if _mode_e6.value == 'FFT-Parameter':
        _cont_e6.children = [_box_fft, _btn_e6, _out_e6]
    else:
        _cont_e6.children = [_box_adc, _btn_e6, _out_e6]

def _calc_e6(_=None):
    _out_e6.clear_output()
    with _out_e6:
        try:
            if _mode_e6.value == 'FFT-Parameter':
                fs = p(_fs_in.value)
                N  = int(p(_N_in.value)) if _N_in.value.strip() else None
                if fs and N:
                    r = fft_parameter(fs, N)
                    print(f"  Δf        = {fmt(r['delta_f'], 'Hz')}")
                    print(f"  f_Nyquist = {fmt(r['f_nyquist'], 'Hz')}")
                    print(f"  T_gesamt  = {fmt(r['T_gesamt'], 's')}")
                    if r['warnung_n_pot2']:
                        print("  ⚠ N ist keine Potenz von 2 → FFT langsamer")
            else:
                N  = int(p(_nb_in.value)) if _nb_in.value.strip() else None
                Ur = p(_ur_in.value) or 3.3
                if N:
                    r = adc_kennwerte(N, Ur, Uin=p(_ui_e6.value))
                    print(f"  LSB    = {fmt(r['LSB'], 'V')}  ({r['LSB_mV']:.3f} mV)")
                    print(f"  SNR    = {r['SNR_dB']:.1f} dB")
                    print(f"  Codes  = {r['codes']}")
                    if 'code_decimal' in r:
                        print(f"  Code   = {r['code_decimal']} ({r['code_hex']})")
                        print(f"  Q-Fehl = {fmt(r['quantisierungs_fehler'], 'V')}")
                    if r.get('warnung'): print(f"  ⚠ {r['warnung']}")
        except (ValueError, ZeroDivisionError, TypeError) as e:
            print("⚠", e)

_mode_e6.observe(_update_e6, names='value')
_btn_e6.on_click(_calc_e6)

display(widgets.HBox(
    [widgets.Box([_theory_out], layout=widgets.Layout(**LEFT_STYLE)),
     widgets.Box([widgets.VBox([_mode_e6, _cont_e6])],
                  layout=widgets.Layout(**RIGHT_STYLE))],
    layout=widgets.Layout(width='100%', align_items='flex-start', margin='8px 0 28px 0')
))
